In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification

# Step 1: Generate a Synthetic Dataset
X, y = make_classification(
    n_samples=1000,       # Number of samples
    n_features=20,        # Total number of features
    n_informative=10,     # Number of informative features
    n_redundant=5,        # Number of redundant features
    random_state=42
)

# Step 2: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Initialize Base Models
lasso = Lasso(alpha=0.01, random_state=42)  # Lasso Regression (embedded feature selection)
gradient_boost = GradientBoostingClassifier(n_estimators=100, random_state=42)  # Gradient Boosting

# Step 4: Perform Blending with K-Fold Cross Validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Create arrays to store predictions from base models
train_meta_features = np.zeros((X_train.shape[0], 2))  # One column for each model's predictions
test_meta_features = np.zeros((X_test.shape[0], 2))    # For test set aggregation

# Train base models and collect predictions
for i, (train_index, val_index) in enumerate(kf.split(X_train)):
    X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]
    y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]

    # Train Lasso Regression
    lasso.fit(X_fold_train, y_fold_train)
    train_meta_features[val_index, 0] = lasso.predict(X_fold_val)
    test_meta_features[:, 0] += lasso.predict(X_test) / n_splits  # Average predictions across folds

    # Train Gradient Boosting
    gradient_boost.fit(X_fold_train, y_fold_train)
    train_meta_features[val_index, 1] = gradient_boost.predict_proba(X_fold_val)[:, 1]
    test_meta_features[:, 1] += gradient_boost.predict_proba(X_test)[:, 1] / n_splits

# Step 5: Train Meta-Model
meta_model = LogisticRegression(random_state=42)
meta_model.fit(train_meta_features, y_train)

# Step 6: Evaluate the Final Blended Model
y_pred_meta = meta_model.predict(test_meta_features)
accuracy = accuracy_score(y_test, y_pred_meta)

print(f"Blended Model Accuracy: {accuracy * 100:.2f}%")
